# Geometry 04 — Raisonner comme un géomètre (DD + AR)

**Public : Licence** — quatrième notebook de la série [Geometry](README.md), le programme gradué de la preuve automatique en géométrie ; il suppose la lecture de 01 (*De la figure à l'équation*) et de 03 (*La méthode de Wu*).

**Ce que ce notebook suppose.** Les prédicats géométriques et leur traduction polynomiale (01), l'idée qu'une méthode de preuve produit une *garantie* vérifiable (02, 03), et le C# de base (tuples, ensembles, fonctions d'ordre supérieur). Aucune connaissance d'AlphaGeometry n'est requise : le notebook la construit.

***

En 2024, le système **AlphaGeometry** (Trinh et al., *Solving olympiad geometry without human demonstrations*) résout des problèmes de géométrie de niveau olympiades sans aucun exemple humain. Sa moitié symbolique — la seule que ce notebook construit — n'est **ni** Gröbner (02), **ni** Wu (03) : c'est un duo inédit dans la série :

- **DD (Deductive Database)** : une base de faits géométriques close sous un corpus de règles élémentaires — la machine *raconte* comme un géomètre, chaque fait dérivé portant sa chaîne de règles ;
- **AR (Algebraic Reasoning)** : la traduction polynomiale de 01/02 réutilisée comme oracle — la machine *calcule* ce que les règles ne joignent pas.

Le fil rouge de la série (le milieu de l'hypoténuse équidistant des trois sommets) reçoit ici son **quatrième regard** — et ce regard a une surprise : DD, laissé seul, *échoue* sur le théorème le plus simple de la série. Comprendre pourquoi est exactement comprendre ce que AlphaGeometry ajoute aux règles.

## §1. Une base de déduction : des faits, des règles, une fermeture

Un **fait** est un tuple `(nom, *points)` : `("midp", "M", "B", "C")` dit que M est le milieu de [BC], `("cong", "A", "M", "B", "M")` que AM = BM, `("perp", "B", "A", "C", "A")` que les droites (AB) et (AC) sont perpendiculaires. La géométrie est symétrique — AM = BM se lit aussi BM = AM — donc chaque fait possède une **forme canonique** : toutes les écritures équivalentes d'un même énoncé tombent sur le même tuple, et la base peut être un simple dictionnaire.

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;
using System.Globalization;
using System.Numerics;

CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.CurrentUICulture = CultureInfo.InvariantCulture;

// Les composantes d'un fait sont réunies par un séparateur pour former une clé de valeur.
string canon(params string[] fact)
{
    var name = fact[0];
    var args = fact.Skip(1).ToArray();
    if (name == "cong") // |AB| = |CD| : aretes non orientees, paire non ordonnee
    {
        var e1 = args.Take(2).OrderBy(x => x, StringComparer.Ordinal).ToArray();
        var e2 = args.Skip(2).Take(2).OrderBy(x => x, StringComparer.Ordinal).ToArray();
        if (StringComparer.Ordinal.Compare(string.Join("\u001f", e1), string.Join("\u001f", e2)) > 0)
            (e1, e2) = (e2, e1);
        return string.Join("\u001f", new[] { name }.Concat(e1).Concat(e2));
    }
    if (name == "coll") // alignement : 3 points non ordonnes
        return string.Join("\u001f", new[] { name }.Concat(args.OrderBy(x => x, StringComparer.Ordinal)));
    if (name == "perp" || name == "para") // droites non orientees, paire non ordonnee
    {
        var l1 = args.Take(2).OrderBy(x => x, StringComparer.Ordinal).ToArray();
        var l2 = args.Skip(2).Take(2).OrderBy(x => x, StringComparer.Ordinal).ToArray();
        if (StringComparer.Ordinal.Compare(string.Join("\u001f", l1), string.Join("\u001f", l2)) > 0)
            (l1, l2) = (l2, l1);
        return string.Join("\u001f", new[] { name }.Concat(l1).Concat(l2));
    }
    if (name == "midp") // M milieu de [AB] : [AB] non oriente
        return string.Join("\u001f", new[] { name, args[0] }.Concat(args.Skip(1).OrderBy(x => x, StringComparer.Ordinal)));
    return string.Join("\u001f", fact);
}

string pretty(string fact)
{
    var f = fact.Split('\u001f');
    switch (f[0])
    {
        case "cong": return $"|{f[1]}{f[2]}| = |{f[3]}{f[4]}|";
        case "coll": return $"{f[1]}, {f[2]}, {f[3]} alignes";
        case "perp": return $"({f[1]}{f[2]}) _|_ ({f[3]}{f[4]})";
        case "para": return $"({f[1]}{f[2]}) // ({f[3]}{f[4]})";
        case "midp": return $"{f[1]} milieu de [{f[2]}{f[3]}]";
        default: return "(" + string.Join(", ", f) + ")";
    }
}

// verifions les symetries sur un exemple : toutes ces ecritures donnent le meme fait
var variants = new[]
{
    new[] { "cong", "A", "M", "B", "M" },
    new[] { "cong", "M", "A", "M", "B" },
    new[] { "cong", "B", "M", "A", "M" },
    new[] { "cong", "M", "B", "M", "A" }
};
var faitsCanoniques = variants.Select(v => canon(v)).ToHashSet();
var rendus = faitsCanoniques.Select(pretty).ToHashSet();
if (faitsCanoniques.Count != 1 || !rendus.SetEquals(new[] { "|AM| = |BM|" }))
    throw new Exception("Écart de canonisation par rapport à la référence Python citée.");
Console.WriteLine("{'" + string.Join("', '", rendus) + "'}");
Console.WriteLine("1 seul fait canonique : " + (faitsCanoniques.Count == 1));

{'|AM| = |BM|'}


1 seul fait canonique : True


### Lecture du résultat

La sortie répond en deux lignes : l'ensemble imprimé est `{'|AM| = |BM|'}` — un seul élément — et le test affiche `1 seul fait canonique : True`. Les quatre variantes de la cellule — `("cong","A","M","B","M")`, `("cong","M","A","M","B")`, `("cong","B","M","A","M")`, `("cong","M","B","M","A")` — se réduisent donc au même fait canonique, rendu `|AM| = |BM|`. C'est tout le propos de `canon` : une arête non orientée (|AM| = |BM| se lit aussi |BM| = |AM|) et une paire non ordonnée ne doivent pas créer de fait nouveau. La conséquence pratique est que la base de faits peut être un simple dictionnaire indexé par le tuple canonique : chaque énoncé géométrique n'y apparaît qu'une fois, quelle que soit sa variante d'écriture. C'est ce qui rend la fermeture du §1 correcte (aucun doublon) et la trace de preuve lisible (un fait = une entrée = une règle et des parents).

Une **règle** est une fonction qui regarde la base et propose de nouveaux faits. Le corpus ci-dessous est volontairement élémentaire — cinq règles qu'un élève de lycée reconnaît :

| Règle | Énoncé | Nature |
|---|---|---|
| `midp_def` | M milieu de [AB] ⇒ AM = MB | définition |
| `midp_coll` | M milieu de [AB] ⇒ A, M, B alignés | définition |
| `cong_trans` | AB = CD et CD = EF ⇒ AB = EF | transitivité |
| `perp_para` | deux perpendiculaires à une même droite ⇒ parallèles | propriété de l'angle droit |
| `coll_perp` | l'alignement propage la perpendicularité | substitution de points |

In [2]:
// Chaque règle renvoie les faits proposés avec son nom et les faits parents.
Dictionary<string, (string rule, string[] parents)> rule_midp_def(Dictionary<string, (string rule, string[] parents)> basis)
{
    var output = new Dictionary<string, (string rule, string[] parents)>();
    foreach (var fact in basis.Keys)
    {
        var f = fact.Split('\u001f');
        if (f[0] == "midp")
            output[canon("cong", f[2], f[1], f[1], f[3])] = ("midp_def", new[] { fact });
    }
    return output;
}

// Un milieu appartient à la droite de ses extrémités.
Dictionary<string, (string rule, string[] parents)> rule_midp_coll(Dictionary<string, (string rule, string[] parents)> basis)
{
    var output = new Dictionary<string, (string rule, string[] parents)>();
    foreach (var fact in basis.Keys)
    {
        var f = fact.Split('\u001f');
        if (f[0] == "midp")
            output[canon("coll", f[2], f[1], f[3])] = ("midp_coll", new[] { fact });
    }
    return output;
}

// Relier les segments par les égalités existantes, puis parcourir les chemins de longueur deux.
Dictionary<string, (string rule, string[] parents)> rule_cong_trans(Dictionary<string, (string rule, string[] parents)> basis)
{
    var output = new Dictionary<string, (string rule, string[] parents)>();
    var byEdge = new Dictionary<(string, string), HashSet<(string fact, (string, string) edge)>>();
    foreach (var fact in basis.Keys)
    {
        var f = fact.Split('\u001f');
        if (f[0] != "cong") continue;
        var e1 = (f[1], f[2]);
        var e2 = (f[3], f[4]);
        if (!byEdge.ContainsKey(e1)) byEdge[e1] = new HashSet<(string, (string, string))>();
        if (!byEdge.ContainsKey(e2)) byEdge[e2] = new HashSet<(string, (string, string))>();
        byEdge[e1].Add((fact, e2));
        byEdge[e2].Add((fact, e1));
    }
    foreach (var entry in byEdge)
        foreach (var first in entry.Value)
            foreach (var second in byEdge[first.edge])
                if (second.edge != entry.Key)
                    output[canon("cong", entry.Key.Item1, entry.Key.Item2, second.edge.Item1, second.edge.Item2)] = ("cong_trans", new[] { first.fact, second.fact });
    return output;
}

// Deux droites perpendiculaires à une même droite sont parallèles.
// La droite commune est indexée : elle peut occuper l'une ou l'autre position dans chaque fait.
Dictionary<string, (string rule, string[] parents)> rule_perp_para(Dictionary<string, (string rule, string[] parents)> basis)
{
    var output = new Dictionary<string, (string rule, string[] parents)>();
    var byPivot = new Dictionary<(string, string), HashSet<(string fact, (string, string) line)>>();
    foreach (var fact in basis.Keys)
    {
        var f = fact.Split('\u001f');
        if (f[0] != "perp") continue;
        var l1 = (f[1], f[2]);
        var l2 = (f[3], f[4]);
        if (!byPivot.ContainsKey(l1)) byPivot[l1] = new HashSet<(string, (string, string))>();
        if (!byPivot.ContainsKey(l2)) byPivot[l2] = new HashSet<(string, (string, string))>();
        byPivot[l1].Add((fact, l2));
        byPivot[l2].Add((fact, l1));
    }
    foreach (var pivot in byPivot)
        foreach (var first in pivot.Value)
            foreach (var second in pivot.Value)
                if (second.fact != first.fact && second.line != first.line)
                    output[canon("para", first.line.Item1, first.line.Item2, second.line.Item1, second.line.Item2)] = ("perp_para", new[] { first.fact, second.fact });
    return output;
}

// Propager une perpendicularité en remplaçant un point par un point aligné.
Dictionary<string, (string rule, string[] parents)> rule_coll_perp(Dictionary<string, (string rule, string[] parents)> basis)
{
    var output = new Dictionary<string, (string rule, string[] parents)>();
    var colls = basis.Keys.Where(f => f.Split('\u001f')[0] == "coll").ToArray();
    var perps = basis.Keys.Where(f => f.Split('\u001f')[0] == "perp").ToArray();
    foreach (var fc in colls)
    {
        var points = fc.Split('\u001f').Skip(1).ToHashSet();
        foreach (var fp in perps)
        {
            var f = fp.Split('\u001f');
            foreach (var pair in new[] { (1, 2), (3, 4) })
            {
                var p = f[pair.Item1];
                var q = f[pair.Item2];
                if (!points.Contains(p) || !points.Contains(q)) continue;
                foreach (var r in points.Where(x => x != p && x != q))
                {
                    var other1 = pair.Item1 == 1 ? f[3] : f[1];
                    var other2 = pair.Item1 == 1 ? f[4] : f[2];
                    output[canon("perp", r, q, other1, other2)] = ("coll_perp", new[] { fc, fp });
                }
            }
        }
    }
    return output;
}

var RULES = new List<Func<Dictionary<string, (string rule, string[] parents)>, Dictionary<string, (string rule, string[] parents)>>>
{
    rule_midp_def, rule_midp_coll, rule_cong_trans, rule_perp_para, rule_coll_perp
};
if (RULES.Count != 5) throw new Exception("Écart sur le nombre de règles : référence Python citée = 5.");

// Témoin de la règle : sur quatre bases de deux faits, la droite commune (M, N) occupe
// chacune des quatre positions possibles — première ou seconde dans chaque fait.
// Le dernier cas est celui qui échappe à un parcours limité à la première position.
int temoinPositions = 0;
var temoinCouvertes = new HashSet<(int, int)>();
var temoinCas = new (string[] faits, string attendu)[]
{
    (new[] { "perp M N P Q", "perp M N Y Z" }, "para P Q Y Z"),
    (new[] { "perp M N P Q", "perp A B M N" }, "para A B P Q"),
    (new[] { "perp A B M N", "perp M N P Q" }, "para A B P Q"),
    (new[] { "perp A B M N", "perp C D M N" }, "para A B C D"),
};
var temoinPartage = ("M", "N");
foreach (var temoinCasCourant in temoinCas)
{
    var temoinBase = new Dictionary<string, (string rule, string[] parents)>();
    foreach (var temoinFait in temoinCasCourant.faits)
        temoinBase[canon(temoinFait.Split(' '))] = ("hypothese", Array.Empty<string>());
    var temoinSortie = rule_perp_para(temoinBase);
    var temoinCle = canon(temoinCasCourant.attendu.Split(' '));
    if (!temoinSortie.ContainsKey(temoinCle) || temoinSortie[temoinCle].rule != "perp_para")
        throw new Exception("Écart de la règle perp_para par rapport à la référence Python citée.");
    // La position lue est celle du fait canonique, pas celle de la saisie.
    var temoinPlacement = new List<int>();
    foreach (var temoinFait in temoinCasCourant.faits)
    {
        var temoinF = canon(temoinFait.Split(' ')).Split('\u001f');
        temoinPlacement.Add((temoinF[1], temoinF[2]) == temoinPartage ? 1 : 2);
    }
    temoinCouvertes.Add((temoinPlacement[0], temoinPlacement[1]));
    temoinPositions++;
}
if (!temoinCouvertes.SetEquals(new[] { (1, 1), (1, 2), (2, 1), (2, 2) }))
    throw new Exception("Écart sur les positions couvertes par la règle perp_para.");

Console.WriteLine($"corpus : {RULES.Count.ToString(CultureInfo.InvariantCulture)} regles elementaires");
Console.WriteLine($"temoin perp_para : {temoinPositions.ToString(CultureInfo.InvariantCulture)} positions sur 4");

corpus : 5 regles elementaires


temoin perp_para : 4 positions sur 4


### Lecture du résultat

Deux lignes : `corpus : 5 regles elementaires`, puis `temoin perp_para : 4 positions sur 4`. Le compte est celui de la liste `RULES` : exactement les cinq règles du tableau de la cellule précédente (`rule_midp_def`, `rule_midp_coll`, `rule_cong_trans`, `rule_perp_para`, `rule_coll_perp`) — aucune oubliée dans le code, aucune ajoutée. Ce qui fait tenir ce corpus, ce n'est pas le nombre mais la forme commune de ses règles : chacune renvoie un dictionnaire `fait -> (règle, faits parents)`. Ce sont ces parents qui fondent la nouveauté pédagogique annoncée : quand le moteur du §1 intégrera ces sorties dans la base, chaque fait dérivé retiendra la règle qui l'a produit et les faits dont elle est partie. La preuve cessera d'être un saut de l'hypothèse à la conclusion pour devenir une chaîne remontable, que `proof_chain` lira dans l'autre sens.

Le **moteur** est une fermeture : on applique toutes les règles jusqu'à ce qu'aucun fait nouveau n'apparaisse (point fixe). La nouveauté pédagogique — c'est elle qui distingue DD d'un simple solveur — est la **trace de preuve** : chaque fait dérivé retient la règle qui l'a produit et les faits parents. La preuve n'est plus un saut no man's land de l'hypothèse à la conclusion : c'est une chaîne qu'on peut remonter et *lire*.

La seconde ligne est le témoin de `rule_perp_para`. Deux faits `perp` qui partagent une même droite donnent deux droites parallèles, et cette droite commune peut tomber en première ou en seconde position dans l'un comme dans l'autre fait : la règle indexe les segments par droite commune, si bien que les quatre placements possibles se dérivent tous. Le témoin les exécute sur des bases de deux hypothèses, hors du fil rouge — le fil rouge ne compte qu'un seul fait `perp`, donc il ne déclenche jamais cette règle et ne dirait rien de sa correction.

In [3]:
// Chaînage avant : les hypothèses et chaque fait dérivé gardent leur provenance.
(Dictionary<string, (string rule, string[] parents)> basis, int rounds) closure(
    IEnumerable<string[]> facts,
    IEnumerable<Func<Dictionary<string, (string rule, string[] parents)>, Dictionary<string, (string rule, string[] parents)>>> rules,
    int max_rounds = 20)
{
    var basis = new Dictionary<string, (string rule, string[] parents)>();
    foreach (var fact in facts) basis[canon(fact)] = ("hypothese", Array.Empty<string>());
    int rounds = 0;
    bool changed = true;
    while (changed && rounds < max_rounds)
    {
        changed = false;
        rounds++;
        foreach (var rule in rules)
            foreach (var entry in rule(basis))
                if (!basis.ContainsKey(entry.Key))
                {
                    basis[entry.Key] = entry.Value;
                    changed = true;
                }
    }
    return (basis, rounds);
}

// Remonter les parents sans répéter un fait déjà rencontré.
List<(string rule, string fact, string[] parents)> proof_chain(
    Dictionary<string, (string rule, string[] parents)> basis, string fact)
{
    var chain = new List<(string rule, string fact, string[] parents)>();
    var seen = new HashSet<string>();
    var stack = new Stack<string>();
    stack.Push(fact);
    while (stack.Count > 0)
    {
        var current = stack.Pop();
        if (!seen.Add(current)) continue;
        var provenance = basis[current];
        chain.Add((provenance.rule, current, provenance.parents));
        foreach (var parent in provenance.parents) stack.Push(parent);
    }
    return chain;
}
Console.WriteLine("moteur DD pret : closure() + proof_chain()");

moteur DD pret : closure() + proof_chain()


### Lecture du résultat

La sortie n'est qu'un fanion — `moteur DD pret : closure() + proof_chain()` — mais elle annonce les deux briques que le §2 va lancer. `closure(facts, rules, max_rounds=20)` fait le chaînage avant jusqu'au point fixe : à chaque tour, toutes les règles parcourent la base et tout fait nouveau y est intégré avec sa provenance — `("hypothese", ())` pour les faits de départ, `(règle, parents)` pour les dérivés — ; elle renvoie `(base, rounds)`. `proof_chain(base, fact)` remonte cette provenance : depuis le fait cible, il empile les parents et rend la liste `(règle, fait, parents)` jusqu'aux hypothèses. Le point fixe garantit la terminaison (au plus `max_rounds` tours, et un fait déjà présent n'est pas réajouté) ; la trace garantit la lisibilité : on pourra relire la preuve du fil rouge comme un texte, règle par règle — c'est exactement ce que l'exercice 2 demandera de faire à la main.

## §2. Le fil rouge, quatrième regard — et un échec instructif

Le théorème : *dans un triangle ABC rectangle en A, le milieu M de l'hypoténuse [BC] est équidistant des trois sommets*. En prédicats, l'hypothèse et la cible :

- **Hypothèses** : `(BA) ⊥ (CA)` et `M milieu de [BC]` ;
- **Cible** : `MA = MB` et `MA = MC` (le fait `MB = MC`, lui, n'est que la définition du milieu).

Lançons la fermeture et regardons *tout ce que DD sait faire* sur ce théorème.

In [4]:
var hypotheses = new[]
{
    new[] { "perp", "B", "A", "C", "A" }, // (BA) _|_ (CA) : angle droit en A
    new[] { "midp", "M", "B", "C" }         // M milieu de l'hypotenuse [BC]
};

var fermeture = closure(hypotheses, RULES);
var basis = fermeture.basis;
var rounds = fermeture.rounds;
// La référence Python citée contient exactement ces faits et provenances.
var attendusDD = new Dictionary<string, string>
{
    [canon("coll", "B", "C", "M")] = "midp_coll",
    [canon("cong", "B", "M", "C", "M")] = "midp_def",
    [canon("midp", "M", "B", "C")] = "hypothese",
    [canon("perp", "B", "A", "C", "A")] = "hypothese"
};
if (rounds != 2 || basis.Count != attendusDD.Count ||
    attendusDD.Any(e => !basis.ContainsKey(e.Key) || basis[e.Key].rule != e.Value))
    throw new Exception("Écart de fermeture par rapport à la référence Python citée.");
Console.WriteLine($"fermeture atteinte en {rounds.ToString(CultureInfo.InvariantCulture)} tours : {basis.Count.ToString(CultureInfo.InvariantCulture)} faits");
foreach (var f in basis.Keys.OrderBy(x => x, StringComparer.Ordinal))
    Console.WriteLine($"  [{basis[f].rule,-10}] {pretty(f)}");

fermeture atteinte en 2 tours : 4 faits


  [midp_coll ] B, C, M alignes


  [midp_def  ] |BM| = |CM|


  [hypothese ] M milieu de [BC]


  [hypothese ] (AB) _|_ (AC)


### Lecture du résultat

La fermeture est… maigre. DD dérive correctement le **combinatoire** : `BM = CM` (définition du milieu) et l'alignement de B, M, C. Mais la cible :

- `MA = MB` : **absent** ;
- `MA = MC` : **absent**.

Ce n'est pas un bug du moteur — c'est une **limite de principe**, et elle est le cœur de ce notebook. La preuve synthétique classique du théorème *construit* un point auxiliaire (le symétrique de A par rapport à M, ou le cercle de diamètre [BC]) : elle ajoute un objet au monde. Or une base de déduction ne fait qu'exploiter les objets *déjà présents* — sur nos quatre points A, B, C, M, aucune chaîne de règles élémentaires ne joint MA à MB. C'est précisément pourquoi AlphaGeometry adjoint à DD un *proposeur* de constructions (neuronal chez lui) : quelqu'un doit agrandir le monde.

La deuxième réponse, celle que ce notebook explore, est de changer d'arithmétique : si les règles ne joignent pas MA et MB, **l'algèbre, elle, sait**.

In [5]:
// Les trois cibles sont mises sous la même forme canonique que la base.
var cibles = new Dictionary<string, string>
{
    ["MA = MB"] = canon("cong", "M", "A", "M", "B"),
    ["MA = MC"] = canon("cong", "M", "A", "M", "C"),
    ["MB = MC"] = canon("cong", "M", "B", "M", "C")
};
var statutsAttendus = new Dictionary<string, bool>
{
    ["MA = MB"] = false,
    ["MA = MC"] = false,
    ["MB = MC"] = true
};
foreach (var cible in cibles)
{
    bool derive = basis.ContainsKey(cible.Value);
    if (derive != statutsAttendus[cible.Key])
        throw new Exception("Écart de statut DD pour " + cible.Key);
    var statut = derive ? "DERIVE par DD" : "ABSENT de DD";
    Console.WriteLine($"{cible.Key,-8} : {statut}");
}
// Vérifier la provenance de l'unique cible dérivée jusqu'à l'hypothèse de milieu.
var chaineMilieu = proof_chain(basis, cibles["MB = MC"]);
var faitMilieu = canon("midp", "M", "B", "C");
if (chaineMilieu.Count != 2 || chaineMilieu[0].rule != "midp_def" ||
    chaineMilieu[0].fact != cibles["MB = MC"] ||
    chaineMilieu[0].parents.Length != 1 || chaineMilieu[0].parents[0] != faitMilieu ||
    chaineMilieu[1].rule != "hypothese" || chaineMilieu[1].fact != faitMilieu ||
    chaineMilieu[1].parents.Length != 0)
    throw new Exception("Écart dans la chaîne de preuve du milieu.");

MA = MB  : ABSENT de DD


MA = MC  : ABSENT de DD


MB = MC  : DERIVE par DD


### Lecture du résultat

Le verdict tient en trois lignes : `MA = MB  : ABSENT de DD`, `MA = MC  : ABSENT de DD`, `MB = MC  : DERIVE par DD`. Le test est un simple test d'appartenance — `basis.ContainsKey(fact)` sur la fermeture du §2 — donc « DERIVE par DD » signifie exactement « présent dans le point fixe », pas « prouvable par un autre chemin ». Or le seul fait dérivé, `MB = MC`, n'est que la définition du milieu (`midp_def`) : la cible métrique du théorème reste hors d'atteinte. La fermeture de 4 faits ne contient aucun fait métrique sur A — ce n'est pas un défaut d'implantation, c'est la limite de principe nommée à la lecture précédente : aucune règle élémentaire ne joint MA à MB sans point auxiliaire. C'est ce constat, chiffré ligne par ligne, qui justifie d'appeler l'algèbre au §3 : AR ne complétera pas le corpus de règles, elle changera de registre, de la combinatoire des faits au calcul sur les polynômes.

## §3. AR : l'algèbre en renfort, avec l'outillage de 01 et 02

Le **raisonnement algébrique** de AlphaGeometry traduit l'énoncé en équations — exactement le geste du notebook 01 — puis les combine. Nous utilisons une base de Gröbner rencontrée en 02, calculée ici par un moteur C# de polynômes à coefficients rationnels exacts :

- l'angle droit en A s'écrit $(\vec{AB} \cdot \vec{AC}) = 0$ ;
- le milieu s'écrit $M = \frac{B+C}{2}$, coordonnée par coordonnée.

La cible $MA^2 = MB^2$ est prouvée si le polynôme $MA^2 - MB^2$ se réduit à zéro modulo les hypothèses.

In [6]:
// Un rationnel est toujours normalisé et son dénominateur est strictement positif.
public readonly struct Rational : IEquatable<Rational>
{
    public BigInteger N { get; }
    public BigInteger D { get; }
    public Rational(BigInteger n, BigInteger d)
    {
        if (d.IsZero) throw new DivideByZeroException();
        if (d.Sign < 0) { n = -n; d = -d; }
        var gcd = BigInteger.GreatestCommonDivisor(BigInteger.Abs(n), d);
        N = n / gcd;
        D = d / gcd;
    }
    public static implicit operator Rational(int n) => new Rational(n, BigInteger.One);
    public static Rational operator +(Rational a, Rational b) => new Rational(a.N * b.D + b.N * a.D, a.D * b.D);
    public static Rational operator -(Rational a, Rational b) => new Rational(a.N * b.D - b.N * a.D, a.D * b.D);
    public static Rational operator -(Rational a) => new Rational(-a.N, a.D);
    public static Rational operator *(Rational a, Rational b) => new Rational(a.N * b.N, a.D * b.D);
    public static Rational operator /(Rational a, Rational b) => new Rational(a.N * b.D, a.D * b.N);
    public bool Equals(Rational other) => N == other.N && D == other.D;
    public override bool Equals(object obj) => obj is Rational other && Equals(other);
    public override int GetHashCode() => HashCode.Combine(N, D);
    public override string ToString() => D == BigInteger.One ? N.ToString(CultureInfo.InvariantCulture) : N.ToString(CultureInfo.InvariantCulture) + "/" + D.ToString(CultureInfo.InvariantCulture);
}

// Les six exposants suivent l'ordre lexicographique xm, ym, xc, yc, xa, ya.
public readonly struct Monomial : IEquatable<Monomial>, IComparable<Monomial>
{
    public int X0 { get; }
    public int X1 { get; }
    public int X2 { get; }
    public int X3 { get; }
    public int X4 { get; }
    public int X5 { get; }
    public Monomial(int a, int b, int c, int d, int e, int f)
    { X0 = a; X1 = b; X2 = c; X3 = d; X4 = e; X5 = f; }
    public int[] Powers() => new[] { X0, X1, X2, X3, X4, X5 };
    public static Monomial From(int[] p) => new Monomial(p[0], p[1], p[2], p[3], p[4], p[5]);
    public int CompareTo(Monomial other)
    {
        var a = Powers(); var b = other.Powers();
        for (int i = 0; i < 6; i++) if (a[i] != b[i]) return a[i].CompareTo(b[i]);
        return 0;
    }
    public bool Equals(Monomial other) => CompareTo(other) == 0;
    public override bool Equals(object obj) => obj is Monomial other && Equals(other);
    public override int GetHashCode() => HashCode.Combine(X0, X1, X2, X3, X4, X5);
    public bool Divides(Monomial other)
    {
        var a = Powers(); var b = other.Powers();
        for (int i = 0; i < 6; i++) if (a[i] > b[i]) return false;
        return true;
    }
    public static Monomial Multiply(Monomial a, Monomial b)
    {
        var x = a.Powers(); var y = b.Powers();
        for (int i = 0; i < 6; i++) x[i] += y[i];
        return From(x);
    }
    public static Monomial Divide(Monomial a, Monomial b)
    {
        if (!b.Divides(a)) throw new ArgumentException("Monôme non divisible.");
        var x = a.Powers(); var y = b.Powers();
        for (int i = 0; i < 6; i++) x[i] -= y[i];
        return From(x);
    }
    public static Monomial Lcm(Monomial a, Monomial b)
    {
        var x = a.Powers(); var y = b.Powers();
        for (int i = 0; i < 6; i++) x[i] = Math.Max(x[i], y[i]);
        return From(x);
    }
}

// Un polynôme creux ne stocke aucun coefficient nul.
public sealed class Polynomial
{
    public Dictionary<Monomial, Rational> Terms { get; } = new Dictionary<Monomial, Rational>();
    public bool IsZero => Terms.Count == 0;
    public static Polynomial Term(Rational coefficient, Monomial monomial)
    {
        var p = new Polynomial();
        if (!coefficient.N.IsZero) p.Terms[monomial] = coefficient;
        return p;
    }
    public static Polynomial Variable(int index)
    {
        var powers = new int[6]; powers[index] = 1;
        return Term(1, Monomial.From(powers));
    }
    public static Polynomial Constant(Rational c) => Term(c, new Monomial());
    public KeyValuePair<Monomial, Rational> Leading()
    {
        if (IsZero) throw new InvalidOperationException("Le polynôme nul n'a pas de terme dominant.");
        return Terms.OrderByDescending(t => t.Key).First();
    }
    public static Polynomial operator +(Polynomial a, Polynomial b)
    {
        var result = new Polynomial();
        foreach (var t in a.Terms) result.Terms[t.Key] = t.Value;
        foreach (var t in b.Terms)
        {
            Rational previous;
            var coefficient = result.Terms.TryGetValue(t.Key, out previous) ? previous + t.Value : t.Value;
            if (coefficient.N.IsZero) result.Terms.Remove(t.Key);
            else result.Terms[t.Key] = coefficient;
        }
        return result;
    }
    public static Polynomial operator -(Polynomial a, Polynomial b) => a + b.Scale(-1);
    public static Polynomial operator *(Polynomial a, Polynomial b)
    {
        var result = new Polynomial();
        foreach (var x in a.Terms)
            foreach (var y in b.Terms)
                result = result + Term(x.Value * y.Value, Monomial.Multiply(x.Key, y.Key));
        return result;
    }
    public Polynomial Scale(Rational c)
    {
        var result = new Polynomial();
        foreach (var t in Terms)
        {
            var value = t.Value * c;
            if (!value.N.IsZero) result.Terms[t.Key] = value;
        }
        return result;
    }
    public Polynomial Monic() => IsZero ? this : Scale((Rational)1 / Leading().Value);
    public bool SameAs(Polynomial other) => (this - other).IsZero;
    public override string ToString()
    {
        if (IsZero) return "0";
        var names = new[] { "xm", "ym", "xc", "yc", "xa", "ya" };
        var parts = new List<string>();
        foreach (var t in Terms.OrderByDescending(t => t.Key))
        {
            var powers = t.Key.Powers();
            var factors = new List<string>();
            for (int i = 0; i < 6; i++)
                if (powers[i] > 0) factors.Add(powers[i] == 1 ? names[i] : names[i] + "^" + powers[i].ToString(CultureInfo.InvariantCulture));
            var abs = new Rational(BigInteger.Abs(t.Value.N), t.Value.D);
            var mon = string.Join("*", factors);
            var body = factors.Count == 0 ? abs.ToString() : (abs.Equals((Rational)1) ? mon : abs.ToString() + "*" + mon);
            var sign = t.Value.N.Sign < 0 ? "-" : "+";
            parts.Add(parts.Count == 0 ? (sign == "-" ? "-" : "") + body : " " + sign + " " + body);
        }
        return string.Concat(parts);
    }
}

public static class GroebnerEngine
{
    // Division multivariée exacte : chaque étape élimine le terme dominant ou le place dans le reste.
    public static Polynomial Reduce(Polynomial input, IReadOnlyList<Polynomial> divisors)
    {
        var p = input;
        var remainder = new Polynomial();
        while (!p.IsZero)
        {
            var lead = p.Leading();
            bool divided = false;
            foreach (var divisor in divisors)
            {
                if (divisor.IsZero) continue;
                var dl = divisor.Leading();
                if (!dl.Key.Divides(lead.Key)) continue;
                var factor = Polynomial.Term(lead.Value / dl.Value, Monomial.Divide(lead.Key, dl.Key));
                p = p - factor * divisor;
                divided = true;
                break;
            }
            if (!divided)
            {
                var term = Polynomial.Term(lead.Value, lead.Key);
                remainder = remainder + term;
                p = p - term;
            }
        }
        return remainder;
    }
    public static Polynomial SPolynomial(Polynomial a, Polynomial b)
    {
        var la = a.Leading(); var lb = b.Leading();
        var lcm = Monomial.Lcm(la.Key, lb.Key);
        return Polynomial.Term((Rational)1 / la.Value, Monomial.Divide(lcm, la.Key)) * a
             - Polynomial.Term((Rational)1 / lb.Value, Monomial.Divide(lcm, lb.Key)) * b;
    }
    // Buchberger traite toutes les paires, y compris celles créées par un nouveau reste.
    public static List<Polynomial> Compute(IEnumerable<Polynomial> generators)
    {
        var basis = generators.Where(p => !p.IsZero).Select(p => p.Monic()).ToList();
        var pairs = new Queue<(int, int)>();
        for (int i = 0; i < basis.Count; i++)
            for (int j = i + 1; j < basis.Count; j++) pairs.Enqueue((i, j));
        while (pairs.Count > 0)
        {
            var pair = pairs.Dequeue();
            var r = Reduce(SPolynomial(basis[pair.Item1], basis[pair.Item2]), basis);
            if (r.IsZero) continue;
            r = r.Monic();
            int next = basis.Count;
            for (int i = 0; i < next; i++) pairs.Enqueue((i, next));
            basis.Add(r);
        }
        // Supprimer les termes dominants redondants, puis réduire chaque polynôme par les autres.
        var minimal = new List<Polynomial>();
        for (int i = 0; i < basis.Count; i++)
        {
            var lm = basis[i].Leading().Key;
            bool redundant = false;
            for (int j = 0; j < basis.Count; j++)
            {
                if (i == j) continue;
                var other = basis[j].Leading().Key;
                if (other.Divides(lm) && (!lm.Equals(other) || j < i)) { redundant = true; break; }
            }
            if (!redundant) minimal.Add(basis[i]);
        }
        var reduced = new List<Polynomial>();
        for (int i = 0; i < minimal.Count; i++)
        {
            var others = minimal.Where((p, j) => i != j).ToList();
            var r = Reduce(minimal[i], others);
            if (!r.IsZero) reduced.Add(r.Monic());
        }
        return reduced.OrderByDescending(p => p.Leading().Key).ToList();
    }
}

// Vérification élémentaire de la division et de l'algorithme de Buchberger.
var testX = Polynomial.Variable(0);
var testY = Polynomial.Variable(1);
var testBasis = GroebnerEngine.Compute(new[] { testX * testX - testY, testX * testY - Polynomial.Constant(1) });
if (!GroebnerEngine.Reduce(testY * testY - testX, testBasis).IsZero ||
    GroebnerEngine.Reduce(Polynomial.Constant(1), testBasis).IsZero)
    throw new Exception("Échec du test élémentaire du moteur polynomial.");

// A=(xa,ya), B=(0,0) origine, C=(xc,yc), M=(xm,ym) : quatre inconnues suffisent
var xm = Polynomial.Variable(0);
var ym = Polynomial.Variable(1);
var xc = Polynomial.Variable(2);
var yc = Polynomial.Variable(3);
var xa = Polynomial.Variable(4);
var ya = Polynomial.Variable(5);
var half = new Rational(BigInteger.One, new BigInteger(2));
var hyp_perp = xa * (xc - xa) + ya * (yc - ya); // (BA).(CA) = 0
var hyp_mid_x = xm - xc.Scale(half);                // M = (B+C)/2, abscisse
var hyp_mid_y = ym - yc.Scale(half);                // M = (B+C)/2, ordonnee
var G = GroebnerEngine.Compute(new[] { hyp_perp, hyp_mid_x, hyp_mid_y });
// Vérifier l'effectif cité, la réduction des générateurs et le critère de Buchberger.
if (G.Count != 3 || new[] { hyp_perp, hyp_mid_x, hyp_mid_y }.Any(p => !GroebnerEngine.Reduce(p, G).IsZero))
    throw new Exception("Écart de base de Gröbner par rapport à la référence Python citée.");
for (int i = 0; i < G.Count; i++)
    for (int j = i + 1; j < G.Count; j++)
        if (!GroebnerEngine.Reduce(GroebnerEngine.SPolynomial(G[i], G[j]), G).IsZero)
            throw new Exception("Critère de Buchberger non satisfait.");
Console.WriteLine("base de Groebner des hypotheses : ok, " + G.Count.ToString(CultureInfo.InvariantCulture) + " polynomes");

base de Groebner des hypotheses : ok, 3 polynomes


### Lecture du résultat

`base de Groebner des hypotheses : ok, 3 polynomes` : les trois hypothèses — l'angle droit en A (`hyp_perp`) et les deux coordonnées du milieu (`hyp_mid_x`, `hyp_mid_y`) — engendrent un idéal que `GroebnerEngine.Compute` réduit ici en une base de 3 polynômes, en ordre lex. C'est cette base `G` qui jouera le rôle d'oracle AR dans la cellule suivante : `GroebnerEngine.Reduce(cible, G)` rend le reste de la division, et le verdict est « résidu nul » = la cible appartient à l'idéal des hypothèses. Un détail qui compte : les coefficients du calcul sont des rationnels exacts ; une identité polynomiale obtenue ainsi est valable en particulier pour les coordonnées réelles de la figure. Un résidu non nul ne suffit pas à lui seul à exclure toutes les figures réelles : son interprétation exige une étude supplémentaire. L'encodage est celui du notebook 01 (A=(xa,ya), B à l'origine, C=(xc,yc), M=(xm,ym)) : six inconnues, trois équations — le prix de la méthode algébrique reste de poser des coordonnées, là où le noyau de 05 n'en posait aucune.

In [7]:
// Prouver une cible en calculant son reste exact modulo la base de Gröbner.
bool ar_prove(Polynomial cible)
{
    return GroebnerEngine.Reduce(cible, G).IsZero;
}

// Construire les trois distances au carré à partir des coordonnées de la figure.
var MA2 = (xm - xa) * (xm - xa) + (ym - ya) * (ym - ya);
var MB2 = xm * xm + ym * ym;
var MC2 = (xm - xc) * (xm - xc) + (ym - yc) * (ym - yc);

var ciblesAR = new Dictionary<string, Polynomial>
{
    ["MA^2 - MB^2"] = MA2 - MB2,
    ["MA^2 - MC^2"] = MA2 - MC2
};
foreach (var cible in ciblesAR)
{
    bool prouve = ar_prove(cible.Value);
    // La référence Python citée donne un résidu nul pour chacune des deux cibles.
    if (!prouve) throw new Exception("Écart de résidu AR pour " + cible.Key);
    Console.WriteLine($"{cible.Key,-14} : {(prouve ? "PROUVE par AR (residu nul)" : "residu NON NUL")}");
}

MA^2 - MB^2    : PROUVE par AR (residu nul)


MA^2 - MC^2    : PROUVE par AR (residu nul)


### Lecture du résultat

Le duo fonctionne : là où DD s'arrête, AR prouve les deux égalités par simple réduction à zéro. Notez la **division du travail** exacte de AlphaGeometry :

| | DD | AR |
|---|---|---|
| Ce qu'il prouve | les faits *combinatoires* (alignements, moitiés, parallèles) | les relations *métriques* (distances, rapports) |
| La preuve est… | **lisible** : une chaîne de règles qu'un humain relit | **calculée** : une réduction modulo un idéal |
| Ce qu'il exige | un corpus de règles, des objets déjà construits | des coordonnées, un solveur polynomial |

Ce partage n'est pas une commodité d'implémentation : dans AlphaGeometry, il faut les deux, car chacun couvre l'angle mort de l'autre — DD ne rejoint pas le métrique sans construction, AR ne dit rien du combinatoire lisible.

## §4. Témoin négatif : un énoncé faux, rejeté deux fois

La règle de fabrication de la série exige un **énoncé faux que la méthode rejette**. Prenons : *la médiane issue de l'angle droit vaut le quart de la demi-hypoténuse*, c'est-à-dire $MA = \frac{MB}{4}$ — $[MB]$ est la demi-hypoténuse, puisque $M$ est le milieu de $[BC]$, et le quart de cette demi-hypoténuse vaut donc le huitième de $[BC]$. DD ne le dérive pas (il n'est nulle part dans la fermeture — bonne nouvelle) mais un moteur qui échoue silencieusement n'est pas un témoin : c'est AR qui transforme l'absence en **réfutation**, avec un résidu non nul mesuré.

In [8]:
// Enonce faux : MA^2 = MB^2 / 16  (soit MA = MB/4, le quart de la demi-hypotenuse [MB])
var enonce_faux = MA2 - MB2.Scale(new Rational(BigInteger.One, new BigInteger(16)));
var residu = GroebnerEngine.Reduce(enonce_faux, G);

// Référence citée du jumeau Python : 15*xc**2/64 + 15*yc**2/64.
// Les coefficients rationnels sont normalisés par le moteur avant comparaison.
var coefficientReference = new Rational(new BigInteger(15), new BigInteger(64));
var residuReference = (xc * xc + yc * yc).Scale(coefficientReference);
if (residu.IsZero || !residu.SameAs(residuReference))
    throw new Exception("Écart sur le résidu de l'énoncé faux par rapport à la référence Python citée.");

// Le langage DD ne possède pas de prédicat pour un rapport de longueurs :
// aucune entrée de la fermeture ne représente cet énoncé, ce n'est pas une preuve de sa fausseté.
var nomsPredicats = basis.Keys.Select(f => f.Split('\u001f')[0]).ToHashSet();
if (!nomsPredicats.SetEquals(new[] { "coll", "cong", "midp", "perp" }))
    throw new Exception("Écart sur les prédicats de la fermeture DD.");
Console.WriteLine("residu de MA^2 - MB^2/16 modulo les hypotheses :");
Console.WriteLine("  " + residu);
Console.WriteLine("verdict AR : " + (!residu.IsZero ? "RESIDU NON NUL -> enonce REJETE" : "residu nul ?!"));
Console.WriteLine("verdict DD : absent de la fermeture (jamais derive)");

residu de MA^2 - MB^2/16 modulo les hypotheses :


  15/64*xc^2 + 15/64*yc^2


verdict AR : RESIDU NON NUL -> enonce REJETE


verdict DD : absent de la fermeture (jamais derive)


### Lecture du résultat

Le rejet est **double** : DD ne dérive jamais l'énoncé faux (il n'est pas conséquence combinatoire des règles), et AR produit un résidu explicitement non nul — l'énoncé est incompatible avec les hypothèses, pas simplement hors d'atteinte. C'est la différence entre *je ne sais pas le prouver* et *c'est faux* : une base de déduction seule ne fait pas la différence, le duo la fait.

Une remarque d'honnêteté : un résidu non nul à la réduction ne prouve la fausseté que sur la **composante non dégénérée** — le même genre de précaution que les conditions de non-dégénérescence de 02 et 03. Lisons le résidu : $\frac{15}{64}(x_c^2 + y_c^2)$ ne s'annule que si C = B — une figure dégénérée d'hypoténuse nulle. Sur toute figure non dégénérée (C ≠ B), l'énoncé faux est bien réfuté ; et c'est une réfutation *plus forte* qu'un simple échec de preuve.

## §5. Ce que le quatrième regard a appris

Le tableau des quatre regards sur le même théorème :

| Notebook | Regard | Ce qu'il garantit |
|---|---|---|
| 01 | numérique (Schwartz-Zippel) | vrai avec grande probabilité sur figures tirées au hasard |
| 02 | algébrique (Gröbner) | vrai exactement, conditions de non-dégénérescence explicites |
| 03 | algorithmique (Wu) | vrai exactement, non-dégénérescences *produites* par l'élimination |
| **04** | **hybride (DD + AR)** | **vrai exactement, avec preuve combinatoire lisible et réfutation mesurée** |

AlphaGeometry complète ce tableau par la suite : DD+AR y est couplé à un proposeur neuronal de constructions (le point auxiliaire qui manquait au §2), ce qui est l'objet de l'accrétion 04b (IMO-AG-30) — hors chemin principal.

## Exercices

Les trois exercices étendent le corpus ou la chaîne de preuve. Ils sont à compléter ; le notebook doit s'exécuter de bout en bout même non complété.

In [9]:
// Exercice 1 — Elargir le corpus.
// Ajoutez une regle rule_coll_trans : si A,B,C sont alignes et A,B,D aussi,
// alors A,C,D sont alignes (transitivite de l'alignement). Relancez la fermeture
// du fil rouge avec le corpus etendu et affichez le nombre de faits nouveaux.
// TODO etudiant
// Indice : parcourez les faits "coll" de la base ; deux faits partageant deux points
//   donnent un troisieme alignement parmi les points restants (canon() deduplique).
// Etape 1 : ecrire rule_coll_trans(base) sur le modele des regles existantes.
// Etape 2 : RULES_EXT = RULES + [rule_coll_trans] puis closure(hypotheses, RULES_EXT).
Console.WriteLine("Exercice a completer");

Exercice a completer


In [10]:
// Exercice 2 — Lire une preuve DD.
// Affichez la chaine de preuve complete du fait BM = CM dans la base du fil rouge
// (fonction proof_chain), puis reecrivez-la en une phrase Francaise du genre
// "BM = CM car M est le milieu de [BC], hypothese du probleme".
// TODO etudiant
// Indice : proof_chain(base, canon(("cong", "M", "B", "M", "C"))) retourne une liste
//   de (regle, fait, parents) ; pretty(fait) rend chaque fait lisible.
// Etape 1 : recuperer la chaine. Etape 2 : l'afficher ligne par ligne.
Console.WriteLine("Exercice a completer");

Exercice a completer


In [11]:
// Exercice 3 — Refuter par AR.
// Testez l'enonce : "la mediane issue de l'angle droit vaut le TIERS de la demi-hypotenuse",
// soit MA^2 = MB^2 / 9. Refutez-le par reduction, puis verifiez au passage que
// l'enonce correct MA^2 = MB^2 reste prouve (residu nul).
// TODO etudiant
// Indice : meme geste que le temoin negatif du 4 : G.reduce(sp.expand(MA2 - MB2/9)).
// Etape 1 : reduire MA2 - MB2/9. Etape 2 : conclure sur le verdict (residu nul ou non).
Console.WriteLine("Exercice a completer");

Exercice a completer


## Conclusion

Ce notebook a construit la moitié symbolique de AlphaGeometry et l'a confrontée au fil rouge de la série :

| Brique | Rôle | Où |
|---|---|---|
| `canon` / `pretty` | faits géométriques sous forme canonique, symétries neutralisées | §1 |
| corpus de 5 règles | les gestes élémentaires du géomètre | §1 |
| `closure` / `proof_chain` | fermeture à point fixe, preuve lisible remontable | §1 |
| l'échec de DD | le métrique sans construction est hors d'atteinte — la limite de principe | §2 |
| AR (`GroebnerEngine.Compute`) | l'oracle polynomial de 01/02, réduction à zéro | §3 |
| témoin négatif | absence (DD) + résidu non nul (AR) = réfutation, pas seulement échec | §4 |

La série poursuit : 04b (IMO-AG-30, le proposeur neuronal couplé à DD+AR) en accrétion, et 05 (un théorème de la série énoncé et prouvé en Lean/Mathlib — que garantit « prouvé par Gröbner » ?) au bout du programme gradué.

**Série Geometry** : [Geometry-04b — IMO-AG-30 →](Geometry-04b-IMO-AG-30-Python.ipynb) | [jumeau Python](Geometry-04-DD-AR-Python.ipynb)

## À retenir

- **Le langage des faits est un langage d'égalité** : `canon` met les faits sous forme canonique et neutralise les symétries — un seul fait subsiste pour le fil rouge, `{|AM| = |BM|}`.
- **Le corpus fait cinq gestes** : cinq règles élémentaires, et la fermeture atteint son point fixe en **2 tours** avec **4 faits** — dont `|BM| = |CM|`, dérivé par DD.
- **DD ne sait pas ce que vaut une distance** : `MA = MB` et `MA = MC` sont **absents** de la fermeture, alors que `MB = MC` en est dérivé. Le combinatoire synthétique ne porte pas le métrique.
- **AR porte le métrique** : la base de Gröbner des hypothèses tient en **3 polynômes**, et les deux conclusions `MA^2 - MB^2` et `MA^2 - MC^2` sont **prouvées** par AR, résidu nul.
- **L'absence et le résidu non nul, ensemble, réfutent** : pour `MA^2 = MB^2/16`, le résidu vaut `15/64*xc^2 + 15/64*yc^2` — `RESIDU NON NUL -> enonce REJETE` — et DD n'a jamais dérivé le fait. Un seul des deux ne suffirait pas.
- **L'échec de DD est une limite de principe, pas un défaut d'implémentation** : la métrique sans construction est hors d'atteinte du langage des règles — c'est ce que le carnet 04b combine avec Wu.

## Vérifiez votre compréhension

1. La fermeture DD du fil rouge atteint 4 faits en 2 tours, dont `|BM| = |CM|`. Pourquoi `|BM| = |CM|` est-il dérivable alors que `MA = MB` ne l'est pas ?
2. Pour `MA^2 = MB^2/16`, DD ne dérive rien et AR calcule un résidu non nul. Que changerait un énoncé dont le résidu serait nul mais que DD n'aurait pas su produire ?
3. `canon` neutralise les symétries. Quel risque reste-t-il si on oublie de l'appliquer sur une règle nouvelle ?
4. Le carnet conclut que « la métrique sans construction est hors d'atteinte ». Cette phrase est-elle un constat d'outil ou une limite de principe, et comment la distinguer des deux ?

<details>
<summary>Réponses</summary>

1. Les deux égalités reposent sur le même fait de milieu (`M` milieu de `[BC]`) et sur le même axiome d'égalité des demi-droites ; `MA = MB` exigerait en plus une relation de distance que le corpus n'énonce pas — il ne manipule que des segments et des alignements.
2. Rien. Un résidu nul **est** une preuve : la conclusion est dans l'idéal des hypothèses, quelle que soit la méthode qui l'a produite. DD est l'organe qui rend la preuve **lisible et remontable** (`proof_chain`), AR est l'organe qui décide. Un résidu nul sans DD reste une preuve correcte, seulement non lisible.
3. Le risque est le **doublon** : la même règle peut dériver le même fait sous deux formes, la closure grossit, et la lecture d'une preuve devient ambiguë — on ne sait plus quel fait a été utilisé. C'est ce que mesure `proof_chain` : une chaîne lisible suppose des faits canoniques.
4. C'est une **limite de principe** : le carnet la mesure sur le corpus de règles lui-même (l'énoncé `MA = MB` est absent de la fermeture, pas « absent ce jour-là »). Un constat d'outil se dirait par « la règle manquait » ou « l'implémentation est incomplète » — et serait réparable par une règle de plus.

</details>

## Pour aller plus loin

- **Combiner DD, AR et Wu à l'échelle d'un sujet d'olympiade** : [Geometry-04b — IMO-AG-30](Geometry-04b-IMO-AG-30-Python.ipynb) prend le problème IMO 2008 P1B, que le benchmark neuro-symbolique ne résout pas, et le traite avec les moteurs construits ici.
- **L'original dont ce carnet ne prend que la moitié** : DeepMind, *AlphaGeometry* (Nature, 2024) — l'architecture complète, proposeur appris compris, dont ce carnet isole la moitié symbolique (DD + AR).
- **La référence du calcul géométrique** : S.-C. Chou, *Mechanical Geometry Theorem Proving* (1988) — le corpus d'encodages et de règles qui sert de référence au geste DD.
- **Dans ce dépôt** : [Geometry-03 — La méthode de Wu](Geometry-03-Wu-Method-Python.ipynb) fournit l'organe d'élimination que le carnet 04b combine avec DD et AR.

### Parité avec le jumeau Python

Les références ci-dessous sont citées du jumeau Python. Les assertions des sections correspondantes comparent les résultats calculés ici à ces références :

| Section | Grandeurs comparées | Écart accepté |
|---|---|---|
| §1 | Forme canonique et taille du corpus | Aucun |
| §2 | Faits, tours, provenances et statuts des cibles | Aucun |
| §3 | Effectif de la base et résidus des deux cibles | Aucun |
| §4 | Polynôme résiduel normalisé | Aucun |